In [5]:
pip install senticnet

   ---------------------------------------- 0.0/51.9 MB ? eta -:--:--
   -------- ------------------------------- 11.3/51.9 MB 59.2 MB/s eta 0:00:01
   -------------------- ------------------- 27.0/51.9 MB 67.6 MB/s eta 0:00:01
   ---------------------------------- ----- 44.8/51.9 MB 74.5 MB/s eta 0:00:01
   ---------------------------------------  51.6/51.9 MB 76.1 MB/s eta 0:00:01
   ---------------------------------------- 51.9/51.9 MB 58.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import time
from senticnet.senticnet import SenticNet
from sklearn.metrics import classification_report

# Initialize SenticNet
sn = SenticNet()

def classify_with_senticnet(text):
    """
    Returns (subjectivity_label, polarity_label)
    Subjectivity: 0 = Objective, 1 = Subjective
    Polarity: 0 = Negative, 1 = Positive
    """
    words = str(text).lower().split()
    total_polarity = 0
    match_count = 0
    
    for word in words:
        try:
            # SenticNet lookup
            val = float(sn.polarity_value(word))
            total_polarity += val
            match_count += 1
        except KeyError:
            continue # Word not in knowledge base
            
    # Subtask 1: Subjectivity Detection
    subjectivity = 1 if match_count > 0 else 0
    
    # Subtask 2: Polarity Detection
    # If neutral or sum is 0, we default to Negative or Neutral based on your project needs
    polarity = 1 if total_polarity > 0 else 0
    
    return subjectivity, polarity

def run_q4_evaluation():
    # Load your manually labeled data
    df = pd.read_excel('eval.xlsx') # Required: 'text', 'gt_subjectivity', 'gt_polarity'
    
    results_subj = []
    results_pol = []
    
    # Measure Performance (Requirement Q4)
    start_time = time.time()
    
    for index, row in df.iterrows():
        subj, pol = classify_with_senticnet(row['text'])
        results_subj.append(subj)
        results_pol.append(pol)
        
    end_time = time.time()
    
    # --- EVALUATION METRICS ---
    
    print("--- Subtask 1: Subjectivity Detection (SenticNet) ---")
    print(classification_report(df['gt_subjectivity'], results_subj, target_names=['Objective', 'Subjective']))
    
    print("\n--- Subtask 2: Polarity Detection (SenticNet) ---")
    # Only evaluate polarity on records that were actually subjective
    subj_indices = [i for i, x in enumerate(df['gt_subjectivity']) if x == 1]
    y_true_pol = [df['gt_polarity'].iloc[i] for i in subj_indices]
    y_pred_pol = [results_pol[i] for i in subj_indices]
    
    print(classification_report(y_true_pol, y_pred_pol, target_names=['Negative', 'Positive']))
    
    # Calculate Speed (Requirement Q4)
    duration = end_time - start_time
    records_per_sec = len(df) / duration
    print(f"\nProcessing Speed: {records_per_sec:.2f} records/second")



In [4]:
run_q4_evaluation()

--- Subtask 1: Subjectivity Detection (SenticNet) ---
              precision    recall  f1-score   support

   Objective       0.00      0.00      0.00         0
  Subjective       1.00      0.80      0.89      1000

    accuracy                           0.80      1000
   macro avg       0.50      0.40      0.44      1000
weighted avg       1.00      0.80      0.89      1000


--- Subtask 2: Polarity Detection (SenticNet) ---
              precision    recall  f1-score   support

    Negative       0.63      0.52      0.57       500
    Positive       0.59      0.70      0.64       500

    accuracy                           0.61      1000
   macro avg       0.61      0.61      0.60      1000
weighted avg       0.61      0.61      0.60      1000


Processing Speed: 18745.16 records/second


E:\conda\envs\d2l\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\conda\envs\d2l\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\conda\envs\d2l\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
